In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import xgboost as xgb

os.makedirs('../models/hypertension', exist_ok=True)

print("✅ Libraries loaded")
print(f"   XGBoost version: {xgb.__version__}")

✅ Libraries loaded
   XGBoost version: 3.2.0


In [2]:
print("Loading hypertension features...")
df = pd.read_csv('../data/processed/hypertension_features.csv')
print(f"✅ Loaded: {df.shape}")
print(f"\nLabel distribution:")
print(df['hypertension_risk'].value_counts())

Loading hypertension features...
✅ Loaded: (118502, 16)

Label distribution:
hypertension_risk
low       61223
medium    31313
high      25966
Name: count, dtype: int64


In [3]:
feature_cols = [
    'age', 'bmi', 'avg_systolic', 'avg_diastolic',
    'bp_variability', 'avg_glucose', 'sugar_variability',
    'avg_pulse', 'adherence_rate', 'missed_doses',
    'has_chronic_condition', 'physical_activity_score',
    'diet_quality_score', 'stress_score', 'sleep_score'
]

X = df[feature_cols]
y = df['hypertension_risk']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"✅ Features shape: {X.shape}")
print(f"✅ Label classes: {le.classes_}")

✅ Features shape: (118502, 15)
✅ Label classes: ['high' 'low' 'medium']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train set: {X_train_scaled.shape}")
print(f"✅ Test set:  {X_test_scaled.shape}")
print(f"\nTrain label distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   {le.classes_[u]}: {c:,} ({c/len(y_train)*100:.1f}%)")

✅ Train set: (94801, 15)
✅ Test set:  (23701, 15)

Train label distribution:
   high: 20,773 (21.9%)
   low: 48,978 (51.7%)
   medium: 25,050 (26.4%)


In [5]:
print("Training XGBoost hypertension model...")
print("This will take 2-3 minutes...")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    num_class=3,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

print("\n✅ Model trained successfully!")

Training XGBoost hypertension model...
This will take 2-3 minutes...
[0]	validation_0-mlogloss:0.91314
[50]	validation_0-mlogloss:0.14919
[100]	validation_0-mlogloss:0.10780
[150]	validation_0-mlogloss:0.09670
[200]	validation_0-mlogloss:0.09253
[250]	validation_0-mlogloss:0.09083
[299]	validation_0-mlogloss:0.08997

✅ Model trained successfully!


In [6]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')

print(f"📊 Model Performance:")
print(f"   Accuracy:  {accuracy*100:.2f}%")
print(f"   AUC Score: {auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

📊 Model Performance:
   Accuracy:  96.28%
   AUC Score: 0.9972

Classification Report:
              precision    recall  f1-score   support

        high       0.98      0.96      0.97      5193
         low       0.98      0.98      0.98     12245
      medium       0.93      0.93      0.93      6263

    accuracy                           0.96     23701
   macro avg       0.96      0.96      0.96     23701
weighted avg       0.96      0.96      0.96     23701



In [7]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("📊 Feature Importance:")
for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"   {row['feature']:<30} {bar} {row['importance']:.4f}")

print("\n📊 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"   Classes: {list(le.classes_)}")
print(cm)

📊 Feature Importance:
   avg_systolic                   ███████████████████████████████████████ 0.3973
   avg_diastolic                  ███████████████████████ 0.2355
   has_chronic_condition          ███████ 0.0751
   age                            █████ 0.0522
   bp_variability                 ████ 0.0494
   missed_doses                   ████ 0.0448
   adherence_rate                 ████ 0.0403
   bmi                            ███ 0.0350
   avg_glucose                    █ 0.0173
   physical_activity_score        █ 0.0141
   stress_score                   █ 0.0107
   sugar_variability               0.0090
   avg_pulse                       0.0067
   sleep_score                     0.0064
   diet_quality_score              0.0060

📊 Confusion Matrix:
   Classes: ['high', 'low', 'medium']
[[ 4998     0   195]
 [    0 11981   264]
 [  122   301  5840]]


In [8]:
joblib.dump(model, '../models/hypertension/hypertension_model.pkl')
joblib.dump(scaler, '../models/hypertension/hypertension_scaler.pkl')
joblib.dump(le, '../models/hypertension/hypertension_label_encoder.pkl')

metadata = {
    "features": feature_cols,
    "target": "hypertension_risk",
    "classes": list(le.classes_),
    "n_samples_train": int(X_train_scaled.shape[0]),
    "n_samples_test": int(X_test_scaled.shape[0]),
    "accuracy": round(float(accuracy), 4),
    "auc_score": round(float(auc), 4),
    "model_version": "1.0",
    "trained_on": pd.Timestamp.now().strftime("%Y-%m-%d")
}

with open('../models/hypertension/hypertension_features.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ All files saved to models/hypertension/:")
print("   hypertension_model.pkl")
print("   hypertension_scaler.pkl")
print("   hypertension_label_encoder.pkl")
print("   hypertension_features.json")
print(f"\n📊 Final Summary:")
print(f"   Accuracy:  {accuracy*100:.2f}%")
print(f"   AUC Score: {auc:.4f}")
print(f"   Classes:   {list(le.classes_)}")

✅ All files saved to models/hypertension/:
   hypertension_model.pkl
   hypertension_scaler.pkl
   hypertension_label_encoder.pkl
   hypertension_features.json

📊 Final Summary:
   Accuracy:  96.28%
   AUC Score: 0.9972
   Classes:   ['high', 'low', 'medium']
